In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, accuracy_score ,classification_report, f1_score , recall_score , precision_score
from sklearn.feature_selection import SelectKBest, f_classif
from xgboost import XGBClassifier

In [4]:
train = pd.read_csv('./train.csv')
test = pd.read_csv("./test.csv")

FileNotFoundError: [Errno 2] No such file or directory: './train.csv'

In [ ]:
X = train.drop(columns=['id',"target"],axis=1)
y = train["target"]
X_test = test.drop('id',axis=1).copy()

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
scale_pos_weight = y.value_counts()[0] / y.value_counts()[1]
scale_pos_weight

In [ ]:
threshold_values = []
balanced_accuracy_scores = []
accuracy_scores = []
f1_scores = []
recall_scores = []
precision_scores = []
y_pred_test = np.zeros(len(X_test))

In [ ]:
for train_index, val_index in skf.split(X, y):
  X_train , X_val = X.iloc[train_index], X.iloc[val_index]
  y_train , y_val = y.iloc[train_index], y.iloc[val_index]
  scaler = StandardScaler()
  selector = SelectKBest(f_classif, k=500)

  X_train_scaled = scaler.fit_transform(X_train)
  X_val_scaled = scaler.transform(X_val)
  X_test_scaled = scaler.transform(X_test)


  X_train_sel = selector.fit_transform(X_train_scaled, y_train)
  X_val_sel = selector.transform(X_val_scaled)
  X_test_sel = selector.transform(X_test_scaled)
  model = XGBClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.85,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
  )
  model.fit(X_train_sel, y_train)
  y_prob_val = model.predict_proba(X_val_sel)[:, 1]
  best_threshold = 0.5
  best_score = 0
  for threshold in np.linspace(0, 1, 300):
      y_pred_val = (y_prob_val > threshold).astype(int)
      score = balanced_accuracy_score(y_val, y_pred_val)
      if score > best_score:
          best_score = score
          best_threshold = threshold
  print(f"Best threshold: {best_threshold}")
  print(f"Best balanced accuracy score: {best_score}")

  y_pred_val = (y_prob_val > best_threshold).astype(int)
  recall_scores.append(recall_score(y_val, y_pred_val))
  precision_scores.append(precision_score(y_val, y_pred_val))
  accuracy_scores.append(accuracy_score(y_val, y_pred_val))
  f1_scores.append(f1_score(y_val, y_pred_val))
  threshold_values.append(best_threshold)
  balanced_accuracy_scores.append(best_score)
  y_pred_test += model.predict_proba(X_test_sel)[:, 1] / skf.n_splits

In [ ]:
final_threshold = np.mean(threshold_values)
print(f"Final threshold: {final_threshold}")
print(f"Final balanced accuracy score: {np.mean(balanced_accuracy_scores)}")
print(f"Final accuracy score: {np.mean(accuracy_scores)}")
print(f"Final f1 score: {np.mean(f1_scores)}")
print(f"Final recall score: {np.mean(recall_scores)}")
print(f"Final precision score: {np.mean(precision_scores)}")

In [ ]:
y_pred_test = (y_pred_test > final_threshold).astype(int)

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "target": y_pred_test
})

In [ ]:
submission.to_csv("submission_anova_xgb_k500.csv", index=False)